# Veritas PoC — EDA

Lightweight inspection of the collected trajectories and the fast vs. slow
causal scores. Run the pipeline first:

```
python -m agent.runner
python -m interp.attribution_patch
python -m interp.ground_truth_patch
python -m analysis.correlate
```

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

trajectories = json.loads((RESULTS / "trajectories.json").read_text())
fast = json.loads((RESULTS / "fast_scores.json").read_text())
slow = json.loads((RESULTS / "slow_scores.json").read_text())

print(f"{len(trajectories)} trajectories, "
      f"{sum(t['success'] for t in trajectories)} successful")

In [ ]:
# Inspect one trajectory: question, gold answer, per-step text and scores
t = trajectories[0]
print("Q:", t["question"])
print("gold:", t["gold_answer"], "| model:", t["generated_answer"], "| success:", t["success"])
print()
for i, txt in enumerate(t["step_texts"]):
    f = fast[t["id"]][i]
    s = slow[t["id"]][i]
    print(f"step {i+1}: fast={f:+.3f} slow={s:+.3f} | {txt[:70]!r}")

In [ ]:
# Fast vs. slow scatter + Pearson r
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

fa, sa = [], []
for tid in fast:
    n = min(len(fast[tid]), len(slow[tid]))
    fa.extend(fast[tid][:n])
    sa.extend(slow[tid][:n])

r, p = pearsonr(fa, sa)
plt.figure(figsize=(5, 5))
plt.scatter(fa, sa, alpha=0.7)
plt.xlabel("fast (attribution)")
plt.ylabel("slow (activation patching)")
plt.title(f"Pearson r = {r:.3f} (p={p:.3g})")
plt.axhline(0, color="grey", lw=0.6)
plt.axvline(0, color="grey", lw=0.6)
plt.tight_layout()
plt.show()